# 답 가능 분류기 v2 학습

판정까지 끝난 (질문, 표 카드) 쌍으로 학습함. 같은 질문 묶음(대조군·항목 없음 최소쌍)을 한 배치에 넣고, 검증은 종류별로 냄. 자세한 내용은 README.md.

런타임은 GPU(A100 권장)로 바꾸고 위에서부터 차례로 실행함.

In [ ]:
# 1. Drive 연결과 작업 폴더
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/colab_project/kosis-answerability-train'  # 이 폴더 위치
PREV = '/content/drive/MyDrive/colab_project/kosis-embedder-train'      # 1차 모델·mined 목록이 있는 폴더
os.chdir(ROOT)
print(os.getcwd(), os.listdir('.'))

In [ ]:
# 2. 규칙 점검 (모델 없이 몇 초)
!python answerability_train_v2.py --selftest

In [ ]:
# 3. 학습 데이터 풀기 (PC에서 만든 data/<BUILD>.zip)
BUILD = 'ans-v2'
import zipfile, json
if not os.path.isdir(f'data/{BUILD}'):
    zipfile.ZipFile(f'data/{BUILD}.zip').extractall('data')
print(json.dumps(json.load(open(f'data/{BUILD}/stats.json', encoding='utf-8')), ensure_ascii=False, indent=1))

In [ ]:
# 4. 시작점 가중치 확인 (리랭커 RR-E2s-e2, 시작점 비교는 보류)
RR = f'{PREV}/outputs/rerank_runs/20261110-013128/RR-E2s-e2.pt'
print('있음' if os.path.isfile(RR) else '없음 (경로 확인)', RR)

In [ ]:
# 5. 빠른 점검: 묶음 200개, 업데이트 20번 (메모리·속도 확인, 몇 분)
!python answerability_train_v2.py --data data/{BUILD} --init rr --init-ckpt "{RR}" --limit 200 --epochs 1 --steps-per-epoch 20 --out outputs/answerability_check

In [ ]:
# 6. 본 학습 (4 epoch). outputs/answerability_runs/<시각>/ANS2-e1.pt ~ e4.pt 저장, 검증이 가장 좋은 epoch을 고름
os.makedirs('outputs', exist_ok=True)
!python answerability_train_v2.py --data data/{BUILD} --init rr --init-ckpt "{RR}" --name ANS2 --epochs 4

In [ ]:
# 7. 결과 요약 (검증셋, 종류별)
import glob
for j in sorted(glob.glob('outputs/answerability_runs/*/ANS2-*.json')):
    m = json.load(open(j, encoding='utf-8'))
    for tag in ('before', f"e{m['epoch']}"):
        v = m['val'][tag]
        print(os.path.relpath(j, 'outputs'), tag, '| AUC', v.get('auc'), '| 최소쌍', v.get('minimal_pair_acc'),
              '| 항목없음 AUC', v.get('auc_pos_vs_no_item'), '| 이웃표 AUC', v.get('auc_pos_vs_neg_near'),
              '| 문항 AUC', v.get('question_auc'), '| 문턱', v.get('question_best_f1', {}).get('threshold'))